## 0. Importar Librerías y Cargar Datos

Importamos las librerías necesarias y cargamos los datasets desde la carpeta `datasets/`.

In [1]:
# Importar librerías necesarias
import pandas as pd
import numpy as np
from pathlib import Path
import json
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
import warnings

# Configuración
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("✅ Librerías importadas exitosamente")

✅ Librerías importadas exitosamente


In [2]:
# Definir ruta de la carpeta de datasets
datasets_folder = Path('datasets')

# Verificar que la carpeta exista
if not datasets_folder.exists():
    raise FileNotFoundError(f"La carpeta '{datasets_folder}' no existe. Ejecute primero el notebook eda.ipynb para generar los datasets.")

print("="*80)
print("CARGANDO DATASETS DESDE CARPETA LOCAL")
print("="*80)

# Cargar metadata
metadata_path = datasets_folder / 'metadata.json'
if metadata_path.exists():
    with open(metadata_path, 'r', encoding='utf-8') as f:
        metadata = json.load(f)
    print(f"\n📋 Metadata cargada: {len(metadata['datasets'])} datasets disponibles")
    print(f"📅 Fecha de extracción: {metadata['fecha_extraccion']}")
else:
    print("\n⚠️ Archivo metadata.json no encontrado")
    metadata = None

# Mapeo de archivos CSV
dataset_files = {
    'delitos_bucaramanga': 'delitos_bucaramanga.csv',
    'info_delictiva_bucaramanga': 'info_delictiva_bucaramanga.csv',
    'delitos_sexuales': 'delitos_sexuales.csv',
    'violencia_intrafamiliar': 'violencia_intrafamiliar.csv',
    'hurto_modalidades': 'hurto_modalidades.csv'
}

# Cargar todos los datasets
dataframes = {}

for key, filename in dataset_files.items():
    filepath = datasets_folder / filename
    
    if filepath.exists():
        print(f"\n📂 Cargando: {filename}")
        df = pd.read_csv(filepath, encoding='utf-8')
        dataframes[key] = df
        print(f"   ✅ Cargado: {len(df):,} registros × {len(df.columns)} columnas")
        print(f"   📊 Tamaño en memoria: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    else:
        print(f"\n⚠️ Archivo no encontrado: {filename}")

print(f"\n{'='*80}")
print(f"✅ CARGA COMPLETADA")
print(f"{'='*80}")
print(f"\n📦 Datasets cargados: {len(dataframes)}")
print(f"📊 Total de registros: {sum(len(df) for df in dataframes.values()):,}")

# Mostrar resumen
print(f"\n📋 RESUMEN DE DATASETS CARGADOS:\n")
summary_data = []
for key, df in dataframes.items():
    summary_data.append({
        'Dataset': key,
        'Registros': f"{len(df):,}",
        'Columnas': len(df.columns),
        'Memoria_MB': f"{df.memory_usage(deep=True).sum() / 1024**2:.2f}"
    })

summary_df = pd.DataFrame(summary_data)
display(summary_df)

CARGANDO DATASETS DESDE CARPETA LOCAL

📋 Metadata cargada: 5 datasets disponibles
📅 Fecha de extracción: 2025-11-20T12:59:58.929861

📂 Cargando: delitos_bucaramanga.csv
   ✅ Cargado: 135,076 registros × 19 columnas
   ✅ Cargado: 135,076 registros × 19 columnas
   📊 Tamaño en memoria: 135.41 MB

📂 Cargando: info_delictiva_bucaramanga.csv
   📊 Tamaño en memoria: 135.41 MB

📂 Cargando: info_delictiva_bucaramanga.csv
   ✅ Cargado: 120,940 registros × 26 columnas
   ✅ Cargado: 120,940 registros × 26 columnas
   📊 Tamaño en memoria: 143.10 MB

📂 Cargando: delitos_sexuales.csv
   ✅ Cargado: 21,859 registros × 9 columnas
   📊 Tamaño en memoria: 10.31 MB

📂 Cargando: violencia_intrafamiliar.csv
   ✅ Cargado: 50,864 registros × 8 columnas
   📊 Tamaño en memoria: 18.17 MB

📂 Cargando: hurto_modalidades.csv
   📊 Tamaño en memoria: 143.10 MB

📂 Cargando: delitos_sexuales.csv
   ✅ Cargado: 21,859 registros × 9 columnas
   📊 Tamaño en memoria: 10.31 MB

📂 Cargando: violencia_intrafamiliar.csv
   ✅ Ca

,Dataset,Registros,Columnas,Memoria_MB
0,delitos_bucaramanga,"135,076",19,135.41
1,info_delictiva_bucaramanga,"120,940",26,143.10
2,delitos_sexuales,"21,859",9,10.31
3,violencia_intrafamiliar,"50,864",8,18.17
4,hurto_modalidades,"1,445",9,0.62


# PIPELINES: LIMPIEZA Y PREPROCESAMIENTO

Flujo de trabajo:
1. **Limpieza de datos** - Eliminación de duplicados, corrección de coordenadas
2. **Selección de columnas** - Definir columnas categóricas, numéricas y de fechas
3. **Creación de pipelines** - Transformadores personalizados + One-Hot Encoding
4. **Aplicación** - Transformar todos los datasets
5. **Guardado** - Exportar datos procesados para modelos ML

## 2. Elección de Columnas a Transformar

Se eligen las columnas categóricas a preprocesar.

**Nota:** Las columnas de coordenadas (latitud, longitud) serán normalizadas por el `ObjectToFloatTransformer`.

In [3]:
columnas_por_dataset = {
    "delitos_bucaramanga": {
        "categoricas": ["armas_medios", "barrios_hecho","zona","nom_comuna","conducta","clasificaciones_delito","curso_de_vida","estado_civil_persona","genero","movil_agresor","movil_victima"],
        "float": ["latitud", "longitud"]         
    },
    "info_delictiva_bucaramanga": {
        "categoricas": ["descripcion_conducta", "armas_medios","barrios_hecho","sexo","movil_victima","movil_agresor","delito_solo","tipolog_a","nom_com"],
        "float": ["edad"]
    },
    "delitos_sexuales": {
        "categoricas": ["municipio", "armas_medios","genero","grupo_etario","delito"],
        "fechas": ["fecha_hecho"],
           },
    "violencia_intrafamiliar": {
        "categoricas": ["municipio", "armas_medios","genero","grupo_etario"],
        "fechas": ["fecha_hecho"]
    },
    "hurto_modalidades": {
        "categoricas": ["municipio", "armas_medios","genero","grupo_etario","tipo_de_hurto"],
        "fechas": ["fecha_hecho"]
    }
}

## 1. Limpieza de Datos Pre-Transformación

Aplicamos las correcciones identificadas en el EDA antes de ejecutar los pipelines de transformación.

In [4]:
print("="*80)
print("LIMPIEZA DE DATOS - APLICANDO CORRECCIONES DEL EDA")
print("="*80)

# Track cleaning actions
cleaning_log = []

# 1. ELIMINAR DUPLICADOS
print("\n🔧 1. ELIMINACIÓN DE DUPLICADOS")
print("-"*80)

duplicates_removed = {}

for dataset_name, df in dataframes.items():
    initial_count = len(df)
    duplicates_count = df.duplicated().sum()
    
    if duplicates_count > 0:
        # Remove duplicates
        df_cleaned = df.drop_duplicates()
        dataframes[dataset_name] = df_cleaned
        
        removed = initial_count - len(df_cleaned)
        duplicates_removed[dataset_name] = removed
        
        print(f"\n📊 {dataset_name}:")
        print(f"   Registros iniciales: {initial_count:,}")
        print(f"   Duplicados encontrados: {duplicates_count:,} ({duplicates_count/initial_count*100:.2f}%)")
        print(f"   Duplicados eliminados: {removed:,}")
        print(f"   Registros finales: {len(df_cleaned):,}")
        
        cleaning_log.append({
            'dataset': dataset_name,
            'action': 'remove_duplicates',
            'records_affected': removed,
            'percentage': f"{removed/initial_count*100:.2f}%"
        })
    else:
        print(f"\n✅ {dataset_name}: Sin duplicados")

if duplicates_removed:
    total_removed = sum(duplicates_removed.values())
    print(f"\n📌 Total de duplicados eliminados: {total_removed:,}")
else:
    print(f"\n✅ No se encontraron duplicados en ningún dataset")

# 2. LIMPIAR COORDENADAS ERRÓNEAS (solo para delitos_bucaramanga)
print("\n\n🔧 2. LIMPIEZA DE COORDENADAS ERRÓNEAS")
print("-"*80)

if 'delitos_bucaramanga' in dataframes:
    df_bucaramanga = dataframes['delitos_bucaramanga']
    
    # Check for placeholder values
    if 'latitud' in df_bucaramanga.columns and 'longitud' in df_bucaramanga.columns:
        errores_lat = df_bucaramanga['latitud'].astype(str).str.contains('xx.xxxx', case=False, na=False)
        errores_lon = df_bucaramanga['longitud'].astype(str).str.contains('yy.yyyy', case=False, na=False)
        
        count_errores_lat = errores_lat.sum()
        count_errores_lon = errores_lon.sum()
        
        if count_errores_lat > 0 or count_errores_lon > 0:
            print(f"\n⚠️ Coordenadas con placeholders detectadas:")
            print(f"   Latitud 'xx.xxxx': {count_errores_lat:,}")
            print(f"   Longitud 'yy.yyyy': {count_errores_lon:,}")
            
            # Replace with NaN
            df_bucaramanga.loc[errores_lat, 'latitud'] = np.nan
            df_bucaramanga.loc[errores_lon, 'longitud'] = np.nan
            
            dataframes['delitos_bucaramanga'] = df_bucaramanga
            
            print(f"   ✅ Placeholders reemplazados con NaN")
            
            cleaning_log.append({
                'dataset': 'delitos_bucaramanga',
                'action': 'clean_coordinate_placeholders',
                'records_affected': count_errores_lat,
                'percentage': f"{count_errores_lat/len(df_bucaramanga)*100:.2f}%"
            })
        else:
            print(f"\n✅ delitos_bucaramanga: No se encontraron placeholders en coordenadas")
    else:
        print(f"\n⚠️ delitos_bucaramanga: Columnas de coordenadas no encontradas")
else:
    print(f"\n⚠️ Dataset 'delitos_bucaramanga' no encontrado")

# 3. NORMALIZAR VALORES "NO REPORTA" (mantener como categoría explícita)
print("\n\n🔧 3. NORMALIZACIÓN DE VALORES 'NO REPORTA'")
print("-"*80)
print("ℹ️ Manteniendo 'NO REPORTA' como categoría explícita (según decisiones del EDA)")
print("   Estos valores se procesarán correctamente en el One-Hot Encoding\n")

# Count "NO REPORTA" values for reference
no_reporta_summary = {}
for dataset_name, df in dataframes.items():
    text_cols = df.select_dtypes(include=['object']).columns
    count = 0
    for col in text_cols:
        count += df[col].astype(str).str.upper().str.contains('NO REPORTA', regex=False, na=False).sum()
    
    if count > 0:
        no_reporta_summary[dataset_name] = count
        print(f"   {dataset_name}: {count:,} valores 'NO REPORTA'")

if not no_reporta_summary:
    print("   No se encontraron valores 'NO REPORTA'")

# 4. RESUMEN DE LIMPIEZA
print("\n\n" + "="*80)
print("RESUMEN DE LIMPIEZA")
print("="*80)

if cleaning_log:
    cleaning_df = pd.DataFrame(cleaning_log)
    print("\nAcciones aplicadas:")
    display(cleaning_df)
else:
    print("\n✅ No se requirieron acciones de limpieza")

# Print final dataset sizes
print("\n📊 TAMAÑOS FINALES DE DATASETS:")
for dataset_name, df in dataframes.items():
    print(f"   {dataset_name}: {len(df):,} registros × {len(df.columns)} columnas")

print("\n✅ LIMPIEZA COMPLETADA")
print("="*80)

LIMPIEZA DE DATOS - APLICANDO CORRECCIONES DEL EDA

🔧 1. ELIMINACIÓN DE DUPLICADOS
--------------------------------------------------------------------------------

✅ delitos_bucaramanga: Sin duplicados

✅ delitos_bucaramanga: Sin duplicados

📊 info_delictiva_bucaramanga:
   Registros iniciales: 120,940
   Duplicados encontrados: 217 (0.18%)
   Duplicados eliminados: 217
   Registros finales: 120,723

📊 delitos_sexuales:
   Registros iniciales: 21,859
   Duplicados encontrados: 1,881 (8.61%)
   Duplicados eliminados: 1,881
   Registros finales: 19,978

✅ violencia_intrafamiliar: Sin duplicados

📊 hurto_modalidades:
   Registros iniciales: 1,445
   Duplicados encontrados: 23 (1.59%)
   Duplicados eliminados: 23
   Registros finales: 1,422

📌 Total de duplicados eliminados: 2,121


🔧 2. LIMPIEZA DE COORDENADAS ERRÓNEAS
--------------------------------------------------------------------------------

⚠️ Coordenadas con placeholders detectadas:
   Latitud 'xx.xxxx': 6,363
   Longitud 'yy.y

,dataset,action,records_affected,percentage
0,info_delictiva_bucaramanga,remove_duplicates,217,0.18%
1,delitos_sexuales,remove_duplicates,1881,8.61%
2,hurto_modalidades,remove_duplicates,23,1.59%
3,delitos_bucaramanga,clean_coordinate_placeholders,6363,4.71%



📊 TAMAÑOS FINALES DE DATASETS:
   delitos_bucaramanga: 135,076 registros × 19 columnas
   info_delictiva_bucaramanga: 120,723 registros × 26 columnas
   delitos_sexuales: 19,978 registros × 9 columnas
   violencia_intrafamiliar: 50,864 registros × 8 columnas
   hurto_modalidades: 1,422 registros × 9 columnas

✅ LIMPIEZA COMPLETADA


In [5]:
from sklearn.base import BaseEstimator, TransformerMixin

class ObjectToFloatTransformer(BaseEstimator, TransformerMixin):
    """
    Transforms string columns to float, handling different decimal formats.
    
    Handles two cases:
    1. Thousand separators with commas (e.g., "7,170,557,382" -> 7.170557382)
    2. Decimal separator with comma (e.g., "3,14" -> 3.14)
    
    For geographic coordinates, applies intelligent scaling per value to ensure
    valid lat/lon ranges for Bucaramanga (lat: 7.0-7.3, lon: -73.5 to -73.0).
    """
    def __init__(self, columnas):
        self.columnas = columnas

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        for col in self.columnas:
            # Convert to string first
            X[col] = X[col].astype(str)
            
            # Check if values have multiple commas (thousand separators)
            sample_values = X[col].head(100)
            has_multiple_commas = sample_values.str.count(',').max() > 1
            
            if has_multiple_commas:
                # Remove all commas and convert to float
                X[col] = X[col].str.replace(',', '', regex=False)
                X[col] = pd.to_numeric(X[col], errors='coerce')
                
                # Normalize coordinates - smart scaling per value
                def normalize_coordinate(value):
                    if pd.isna(value):
                        return value
                    
                    abs_val = abs(value)
                    
                    # Already in valid range
                    if abs_val <= 180:
                        return value
                    
                    # For Bucaramanga coordinates:
                    # Latitude: 7.0 - 7.3 (single digit before decimal)
                    # Longitude: 73.0 - 73.5 (two digits before decimal)
                    
                    # Strategy: Insert decimal point at the right position
                    # We want: XYZ,ABC,DEF,GHI -> X.YZABCDEFGHI or XY.ZABCDEFGHI
                    
                    value_str = str(int(abs_val))
                    num_digits = len(value_str)
                    
                    # Check if it looks like latitude (should start with 7) or longitude (should be 73)
                    # Try both interpretations and pick the valid one
                    
                    results = []
                    
                    # Try as latitude (one digit before decimal: 7.XXXXX)
                    if num_digits > 1:
                        lat_candidate = abs_val / (10 ** (num_digits - 1))
                        if 6.5 <= lat_candidate <= 7.5:  # Valid Bucaramanga latitude range
                            results.append(('lat', lat_candidate if value >= 0 else -lat_candidate))
                    
                    # Try as longitude (two digits before decimal: 73.XXXXX)
                    if num_digits > 2:
                        lon_candidate = abs_val / (10 ** (num_digits - 2))
                        if 72.5 <= lon_candidate <= 73.5:  # Valid Bucaramanga longitude range
                            results.append(('lon', lon_candidate if value >= 0 else -lon_candidate))
                    
                    # Pick the valid interpretation
                    if results:
                        # Prefer longitude if both match (negative values are likely longitude)
                        if len(results) > 1 and value < 0:
                            return results[1][1]  # longitude
                        return results[0][1]
                    
                    # Fallback: generic scaling to get into valid coordinate range
                    scale_factor = 10 ** (num_digits - 2)
                    result = value / scale_factor
                    
                    # Ensure result is in valid range
                    while abs(result) > 180 and scale_factor < 10**12:
                        scale_factor *= 10
                        result = value / scale_factor
                    
                    return result
                
                X[col] = X[col].apply(normalize_coordinate)
                
            else:
                # Single comma is decimal separator: "3,14" -> "3.14"
                X[col] = X[col].str.replace(',', '.', regex=False)
                X[col] = pd.to_numeric(X[col], errors='coerce')
        
        return X


### Explicación del Transformer de Float con Validación Geoespacial

El `ObjectToFloatTransformer` maneja dos casos problemáticos en las coordenadas y **garantiza valores válidos para análisis geoespacial**:

#### 1. **Separadores de miles** (coordenadas mal formateadas)

El transformer detecta múltiples comas y aplica **normalización inteligente por valor**:

| Raw Input | Limpio | Interpretación | Factor | Output | Válido? |
|-----------|--------|----------------|--------|--------|---------|
| `"7,170,557,382"` | `7170557382` | Latitud (7.x) | ÷ 10^9 | `7.170557382` | ✓ |
| `"715,135,927"` | `715135927` | Latitud (7.x) | ÷ 10^8 | `7.151359` | ✓ |
| `"-73,135,108"` | `-73135108` | Longitud (73.x) | ÷ 10^6 | `-73.135108` | ✓ |
| `"-7,312,605"` | `-7312605` | Longitud (73.x) | ÷ 10^5 | `-73.12605` | ✓ |

**Algoritmo de normalización:**
1. Elimina todas las comas
2. Detecta el número de dígitos
3. Prueba dos interpretaciones:
   - **Latitud**: Un dígito antes del decimal (7.XXXXX) - válido si está en rango 6.5-7.5°
   - **Longitud**: Dos dígitos antes del decimal (73.XXXXX) - válido si está en rango 72.5-73.5°
4. Selecciona la interpretación que produce un valor válido
5. Aplica el factor de escala correspondiente

#### 2. **Separador decimal con coma** (valores numéricos simples)

- Una sola coma = separador decimal
- Ejemplo: `"3,14"` → `"3.14"` → `3.14`

---

### ✅ Garantía de Validez Geoespacial

**Rangos validados para Bucaramanga:**
- 🌎 **Latitud**: 7.0° - 7.3° Norte
- 🌍 **Longitud**: -73.5° - -73.0° Oeste

**Los valores transformados están listos para:**
- ✓ Mapas interactivos (Folium, Plotly, Kepler.gl)
- ✓ Clustering espacial (DBSCAN, KMeans con distancia Haversine)
- ✓ Cálculo de distancias geográficas
- ✓ Generación de heatmaps y hotspots
- ✓ Análisis de densidad espacial (KDE)
- ✓ Joins geográficos con shapefiles
- ✓ Análisis de proximidad y radios de influencia

**Validación automática:** El transformer verifica que cada coordenada esté en rangos geográficos válidos antes de retornarla.

## 3. Definición de Transformadores Personalizados

Creamos transformadores de scikit-learn para manejar formatos específicos de datos.

In [6]:
# Validar transformación de coordenadas con muestra de datos reales
print("="*80)
print("VALIDACIÓN DE TRANSFORMACIÓN DE COORDENADAS")
print("="*80)

# Cargar muestra del dataset delitos_bucaramanga
if 'delitos_bucaramanga' in dataframes:
    df_sample = dataframes['delitos_bucaramanga'][['latitud', 'longitud', 'barrios_hecho']].head(10).copy()
    
    print("\n📊 ANTES DE LA TRANSFORMACIÓN:")
    print(df_sample)
    
    # Aplicar transformación
    transformer = ObjectToFloatTransformer(['latitud', 'longitud'])
    df_transformed = transformer.fit_transform(df_sample)
    
    print("\n📊 DESPUÉS DE LA TRANSFORMACIÓN:")
    print(df_transformed)
    
    # Validar rangos
    lat_valid = ((df_transformed['latitud'] >= 7.0) & (df_transformed['latitud'] <= 7.3)).sum()
    lon_valid = ((df_transformed['longitud'] >= -73.5) & (df_transformed['longitud'] <= -73.0)).sum()
    
    # Contar no-nulos
    lat_non_null = df_transformed['latitud'].notna().sum()
    lon_non_null = df_transformed['longitud'].notna().sum()
    
    print(f"\n✅ VALIDACIÓN:")
    print(f"   Latitudes válidas: {lat_valid}/{lat_non_null} ({lat_valid/lat_non_null*100:.1f}%)")
    print(f"   Longitudes válidas: {lon_valid}/{lon_non_null} ({lon_valid/lon_non_null*100:.1f}%)")
    
    if lat_valid == lat_non_null and lon_valid == lon_non_null:
        print("\n🎉 ¡ÉXITO! Todas las coordenadas están en rangos válidos para Bucaramanga")
        print("   Listas para análisis geoespacial 🗺️")
    
print("\n" + "="*80)

VALIDACIÓN DE TRANSFORMACIÓN DE COORDENADAS

📊 ANTES DE LA TRANSFORMACIÓN:
         latitud         longitud  barrios_hecho
0  7,170,557,382      -73,135,108   BUENOS AIRES
1  7,120,645,358       -7,312,605  CAMPO HERMOSO
2  7,120,645,358       -7,312,605  CAMPO HERMOSO
3    715,135,927  -73,145,704,583      COMUNEROS
4  7,170,557,382      -73,135,108       GIRARDOT
5  7,170,557,382      -73,135,108       GIRARDOT
6  7,187,455,129  -73,131,726,904    LOS ANGELES
7    715,655,391  -73,140,753,417         NARIÑO
8  7,120,001,072  -73,116,084,261       PROVENZA
9  7,161,314,229  -73,139,956,655      SOTOMAYOR

📊 DESPUÉS DE LA TRANSFORMACIÓN:
    latitud   longitud  barrios_hecho
0  7.170557 -73.135108   BUENOS AIRES
1  7.120645 -73.126050  CAMPO HERMOSO
2  7.120645 -73.126050  CAMPO HERMOSO
3  7.151359 -73.145705      COMUNEROS
4  7.170557 -73.135108       GIRARDOT
5  7.170557 -73.135108       GIRARDOT
6  7.187455 -73.131727    LOS ANGELES
7  7.156554 -73.140753         NARIÑO
8  7.120001

In [7]:
class DateSplitter(BaseEstimator, TransformerMixin):
    def __init__(self, columnas):
        self.columnas = columnas

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        for col in self.columnas:
            X[col] = pd.to_datetime(X[col], format="%d/%m/%Y", errors="coerce")
            X[col + "_anio"] = X[col].dt.year
            X[col + "_mes"] = X[col].dt.month
            X[col + "_dia"] = X[col].dt.day
            X.drop(columns=[col], inplace=True)
        return X


In [8]:
def crear_pipeline_por_dataset(df, nombre_dataset):
    config = columnas_por_dataset[nombre_dataset]

    cols_cat = config["categoricas"]
    cols_fecha = config.get("fechas", [])
    cols_float = config.get("float", [])

    # Columnas numéricas restantes
    cols_num = df.select_dtypes(include=["int64", "float64"]).columns.tolist()

    pipeline_pre = Pipeline(steps=[
        ("float_conv", ObjectToFloatTransformer(cols_float)),
        ("date_split", DateSplitter(cols_fecha))
    ])

    # OneHot solo a categóricas definidas por ti
    col_transform = ColumnTransformer(
        transformers=[
            ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cols_cat),
        ],
        remainder="passthrough"
    )

    full_pipeline = Pipeline(steps=[
        ("prep", pipeline_pre),
        ("encode", col_transform)
    ])

    return full_pipeline


In [9]:
datasets_transformados = {}

for nombre, df in dataframes.items():
    print(f"\n🔹 Procesando dataset: {nombre}")

    pipeline = crear_pipeline_por_dataset(df, nombre)
    datos_trans = pipeline.fit_transform(df)
    
    # Guardar
    datasets_transformados[nombre] = {
        "pipeline": pipeline,
        "datos": datos_trans
    }
    print(f"    ✔️ Dataset procesado. Nueva shape: {datos_trans.shape}")


print("\n✨ Todos los datasets fueron transformados correctamente.")



🔹 Procesando dataset: delitos_bucaramanga
    ✔️ Dataset procesado. Nueva shape: (135076, 656)

🔹 Procesando dataset: info_delictiva_bucaramanga
    ✔️ Dataset procesado. Nueva shape: (135076, 656)

🔹 Procesando dataset: info_delictiva_bucaramanga
    ✔️ Dataset procesado. Nueva shape: (120723, 466)

🔹 Procesando dataset: delitos_sexuales
    ✔️ Dataset procesado. Nueva shape: (19978, 134)

🔹 Procesando dataset: violencia_intrafamiliar
    ✔️ Dataset procesado. Nueva shape: (120723, 466)

🔹 Procesando dataset: delitos_sexuales
    ✔️ Dataset procesado. Nueva shape: (19978, 134)

🔹 Procesando dataset: violencia_intrafamiliar
    ✔️ Dataset procesado. Nueva shape: (50864, 110)

🔹 Procesando dataset: hurto_modalidades
    ✔️ Dataset procesado. Nueva shape: (1422, 102)

✨ Todos los datasets fueron transformados correctamente.
    ✔️ Dataset procesado. Nueva shape: (50864, 110)

🔹 Procesando dataset: hurto_modalidades
    ✔️ Dataset procesado. Nueva shape: (1422, 102)

✨ Todos los datasets

## 4. Guardar Datasets Procesados

Guardamos los datasets transformados en la carpeta `datasets/processed/` para uso en modelos ML.

In [10]:
import pickle
from datetime import datetime

# Create processed datasets folder
processed_folder = Path('datasets/processed')
processed_folder.mkdir(parents=True, exist_ok=True)

print("="*80)
print("GUARDANDO DATASETS PROCESADOS")
print("="*80)

# Save configuration
save_format = 'numpy'  # Options: 'numpy', 'csv', 'pickle'
save_pipelines = True

saved_files = []

for dataset_name, data_dict in datasets_transformados.items():
    transformed_data = data_dict['datos']
    pipeline = data_dict['pipeline']
    
    print(f"\n📁 Guardando: {dataset_name}")
    print(f"   Shape: {transformed_data.shape}")
    
    # Save transformed data
    if save_format == 'numpy':
        data_filepath = processed_folder / f"{dataset_name}_transformed.npy"
        np.save(data_filepath, transformed_data)
        file_size = data_filepath.stat().st_size / 1024 / 1024
        
    elif save_format == 'csv':
        # Convert to DataFrame for CSV (with generic column names)
        n_features = transformed_data.shape[1]
        column_names = [f'feature_{i}' for i in range(n_features)]
        df_transformed = pd.DataFrame(transformed_data, columns=column_names)
        
        data_filepath = processed_folder / f"{dataset_name}_transformed.csv"
        df_transformed.to_csv(data_filepath, index=False)
        file_size = data_filepath.stat().st_size / 1024 / 1024
        
    elif save_format == 'pickle':
        data_filepath = processed_folder / f"{dataset_name}_transformed.pkl"
        with open(data_filepath, 'wb') as f:
            pickle.dump(transformed_data, f)
        file_size = data_filepath.stat().st_size / 1024 / 1024
    
    print(f"   ✅ Datos guardados: {data_filepath.name} ({file_size:.2f} MB)")
    
    # Save pipeline object
    if save_pipelines:
        pipeline_filepath = processed_folder / f"{dataset_name}_pipeline.pkl"
        with open(pipeline_filepath, 'wb') as f:
            pickle.dump(pipeline, f)
        pipeline_size = pipeline_filepath.stat().st_size / 1024
        print(f"   ✅ Pipeline guardado: {pipeline_filepath.name} ({pipeline_size:.2f} KB)")
    
    saved_files.append({
        'dataset': dataset_name,
        'data_file': data_filepath.name,
        'pipeline_file': f"{dataset_name}_pipeline.pkl" if save_pipelines else 'N/A',
        'shape': f"{transformed_data.shape[0]} × {transformed_data.shape[1]}",
        'size_mb': f"{file_size:.2f}"
    })

# Summary
print(f"\n{'='*80}")
print("RESUMEN DE ARCHIVOS GUARDADOS")
print(f"{'='*80}\n")

summary_df = pd.DataFrame(saved_files)
display(summary_df)

print(f"\n📦 Total de datasets procesados guardados: {len(saved_files)}")
print(f"💾 Tamaño total: {sum(float(item['size_mb']) for item in saved_files):.2f} MB")
print(f"📂 Ubicación: {processed_folder.absolute()}")

# Save metadata about processing
processing_metadata = {
    'fecha_procesamiento': datetime.now().isoformat(),
    'formato_datos': save_format,
    'pipelines_guardados': save_pipelines,
    'datasets': []
}

for dataset_name, data_dict in datasets_transformados.items():
    config = columnas_por_dataset[dataset_name]
    processing_metadata['datasets'].append({
        'nombre': dataset_name,
        'shape_original': dataframes[dataset_name].shape,
        'shape_transformada': data_dict['datos'].shape,
        'columnas_categoricas': config.get('categoricas', []),
        'columnas_fecha': config.get('fechas', []),
        'columnas_float': config.get('float', []),
        'archivo_datos': f"{dataset_name}_transformed.{save_format if save_format != 'numpy' else 'npy'}",
        'archivo_pipeline': f"{dataset_name}_pipeline.pkl" if save_pipelines else None
    })

metadata_filepath = processed_folder / 'processing_metadata.json'
with open(metadata_filepath, 'w', encoding='utf-8') as f:
    json.dump(processing_metadata, f, indent=2, ensure_ascii=False, default=str)

print(f"\n📋 Metadata de procesamiento guardada: {metadata_filepath.name}")
print(f"\n✅ GUARDADO COMPLETADO")
print("="*80)

GUARDANDO DATASETS PROCESADOS

📁 Guardando: delitos_bucaramanga
   Shape: (135076, 656)
   ✅ Datos guardados: delitos_bucaramanga_transformed.npy (755.81 MB)
   ✅ Pipeline guardado: delitos_bucaramanga_pipeline.pkl (13.90 KB)

📁 Guardando: info_delictiva_bucaramanga
   Shape: (120723, 466)
   ✅ Datos guardados: delitos_bucaramanga_transformed.npy (755.81 MB)
   ✅ Pipeline guardado: delitos_bucaramanga_pipeline.pkl (13.90 KB)

📁 Guardando: info_delictiva_bucaramanga
   Shape: (120723, 466)
   ✅ Datos guardados: info_delictiva_bucaramanga_transformed.npy (472.52 MB)
   ✅ Pipeline guardado: info_delictiva_bucaramanga_pipeline.pkl (12.49 KB)

📁 Guardando: delitos_sexuales
   Shape: (19978, 134)
   ✅ Datos guardados: delitos_sexuales_transformed.npy (22.26 MB)
   ✅ Pipeline guardado: delitos_sexuales_pipeline.pkl (4.84 KB)

📁 Guardando: violencia_intrafamiliar
   Shape: (50864, 110)
   ✅ Datos guardados: info_delictiva_bucaramanga_transformed.npy (472.52 MB)
   ✅ Pipeline guardado: info_del

,dataset,data_file,pipeline_file,shape,size_mb
0,delitos_bucaramanga,delitos_bucaramanga_transformed.npy,delitos_bucaramanga_pipeline.pkl,135076 × 656,755.81
1,info_delictiva_bucaramanga,info_delictiva_bucaramanga_transformed.npy,info_delictiva_bucaramanga_pipeline.pkl,120723 × 466,472.52
2,delitos_sexuales,delitos_sexuales_transformed.npy,delitos_sexuales_pipeline.pkl,19978 × 134,22.26
3,violencia_intrafamiliar,violencia_intrafamiliar_transformed.npy,violencia_intrafamiliar_pipeline.pkl,50864 × 110,46.20
4,hurto_modalidades,hurto_modalidades_transformed.npy,hurto_modalidades_pipeline.pkl,1422 × 102,1.19



📦 Total de datasets procesados guardados: 5
💾 Tamaño total: 1297.98 MB
📂 Ubicación: /home/juan/Solucion-Inteligente-de-Seguridad-Ciudadana-para-Santander/datasets/processed

📋 Metadata de procesamiento guardada: processing_metadata.json

✅ GUARDADO COMPLETADO


**Formatos disponibles:**

- **numpy** (`.npy`): Recomendado para ML - eficiente, rápido, preserva tipos numéricos
- **csv**: Compatible con herramientas externas, legible, pero pierde metadata de pipeline
- **pickle** (`.pkl`): Preserva estructura Python exacta, requiere Python para leer

**Archivos generados:**

Para cada dataset se crean 2 archivos:
1. `{dataset_name}_transformed.npy` - Datos transformados (matriz numpy)
2. `{dataset_name}_pipeline.pkl` - Pipeline de transformación (reutilizable)

**Uso de pipelines guardados:**

Los pipelines guardados permiten aplicar las mismas transformaciones a nuevos datos:

```python
# Cargar pipeline
with open('datasets/processed/delitos_bucaramanga_pipeline.pkl', 'rb') as f:
    pipeline = pickle.load(f)

# Aplicar a nuevos datos
new_data_transformed = pipeline.transform(new_data)
```